# SOXX 2019–2022 NPORT-P 原 XML 来源批次

重用已 NAS 验证的首份 2019-09-30 样本，只补其余 12 条 series 专属原始 XML。每份核对 CIK/series/class/repPdDate 与投资条目结构；不读取收益或拟合。


In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex,xml.etree.ElementTree as ET
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2019_2022_nport_original_xml_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
 return h.hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
assert sha(root/plan['parent_index']['path'])==plan['parent_index']['sha256']
assert sha(root/plan['reused_specimen']['schema_result'])==plan['reused_specimen']['schema_result_sha256']
assert sha(root/plan['reused_specimen']['raw_path'])==plan['reused_specimen']['raw_sha256']
raw=root/'data/snapshots/stage1_soxx_2019_2022_nport_original_xml_20261023';raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','GET_attempts':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];error=None
for i,spec in enumerate(plan['exact_additional_XML']):
 state['GET_attempts']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 accession=spec['accession'];body=raw/(accession+'.xml');headers=raw/(accession+'.headers');started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['transport']['connect_timeout_seconds']),'--max-time',str(plan['transport']['total_timeout_seconds']),'--max-filesize',str(plan['transport']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['transport']['user_agent'],'-H','Accept: application/xml','-w','%{http_code} %{url_effective}',spec['url']]
 try:
  run=subprocess.run(cmd,capture_output=True,text=True,timeout=60);fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
  receipt={'accession':accession,'expected_report_period':spec['report_period'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'stderr':run.stderr[:300]}
 except Exception as exc:
  receipt={'accession':accession,'expected_report_period':spec['report_period'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:180]}
 receipt.update(raw_sha256=sha(body) if body.exists() else None,raw_bytes=body.stat().st_size if body.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
 receipts.append(receipt)
 if receipt.get('http_status')!=200 or receipt.get('curl_exit_code')!=0 or not body.exists() or body.stat().st_size>plan['transport']['max_response_bytes_each']:
  error='transport_or_size_failure:'+accession;break
 try:
  xml=ET.fromstring(body.read_bytes());local=lambda t:t.rsplit('}',1)[-1]
  vals=lambda name:sorted({(e.text or '').strip() for e in xml.iter() if local(e.tag)==name and (e.text or '').strip()})
  assert vals('cik')==['0001100663'] and vals('seriesId')==['S000004354'] and vals('classId')==['C000012084']
  assert vals('repPdDate')==[spec['report_period']]
  count=sum(local(e.tag)=='invstOrSec' for e in xml.iter());assert count>0
  receipt['source_identity_passed']=True;receipt['investment_row_count']=count;receipt['report_period_original']=vals('repPdDate')[0]
 except Exception as exc:
  error='XML_or_identity_failure:'+accession+':'+type(exc).__name__+':'+str(exc)[:140];receipt['source_identity_passed']=False;break
 if i+1<len(plan['exact_additional_XML']):time.sleep(plan['transport']['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'name':p.name,'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw.iterdir()) if p.is_file()],'capture_receipts_sha256':sha(card/'capture_receipts.json')}
(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-nport-xml-source-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for x in manifest['files']:out.add(raw/x['name'],arcname='raw/'+x['name'])
  out.add(card/'source_manifest.json',arcname='source_manifest.json');out.add(card/'capture_receipts.json',arcname='capture_receipts.json')
 digest=sha(archive);evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-nport-xml-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for x in manifest['files']:
    member=source.extractfile('raw/'+x['name']);assert member and hashlib.sha256(member.read()).hexdigest()==x['sha256']
result={'status':'twelve_original_NPORT_XML_sources_archived_identity_passed' if error is None and len(receipts)==12 else 'partial_NPORT_XML_source_needs_repair','error':error,'new_GET_attempts':state['GET_attempts'],'new_XML_identity_passed':sum(x.get('source_identity_passed') is True for x in receipts),'reused_2019_specimen_sha256':plan['reused_specimen']['raw_sha256'],'total_distinct_original_XML_sources':1+sum(x.get('source_identity_passed') is True for x in receipts),'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'holding_values_parsed':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'source_result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','new_GET_attempts','new_XML_identity_passed','total_distinct_original_XML_sources','nas_archive_sha256')})


{'status': 'twelve_original_NPORT_XML_sources_archived_identity_passed', 'new_GET_attempts': 12, 'new_XML_identity_passed': 12, 'total_distinct_original_XML_sources': 13, 'nas_archive_sha256': 'a5834ee5d13a560409aca89f6e56d2abc36e099b7dc937e67d67af3697f30641'}


## 中文结论

新增 12 份 SOXX NPORT-P 原 XML 均 HTTP 200 且 CIK/series/class/repPdDate 与各自已封存 filing index 一致；连同独立备份的首份 2019 样本，共 13 份来源结构合格。新增原件 NAS 归档 SHA-256 `a5834ee5d13a560409aca89f6e56d2abc36e099b7dc937e67d67af3697f30641`，远端与定点恢复通过。0 数值因子、0 收益、0 拟合、0 账户；披露滞后和旧年历史仍待处理。
